# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Wajah) & Gaussian Naive Baye

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

C:\Users\Rezky\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images/able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images/able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images/able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images/able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images/able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images/zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images/zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images/zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images/zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
vggface_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(emotion_features[key]) + list(vggface_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|█████████████████████████████████████████████████████████████████████████| 10800/10800 [00:00<00:00, 12847.81it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    }
]

pipeline = Pipeline(steps=[
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
    'roc_auc_ovr': 'roc_auc_ovr'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.8),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 40 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    model_prefix='models/clf_demogpairs_gnb_vit-emotion-face_',
    results_path='results/demogpairs_gnb_vit-emotion-face_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Mengevaluasi model: GaussianNB
Fitting 5 folds for each of 40 candidates, totalling 200 fits


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.151177507061566)}


Accuracy  : 0.825
Precision : 0.824763752562418
Recall    : 0.8250000000000001
F1 Score  : 0.8240249078492402
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       360
           1       0.86      0.89      0.87       360
           2       0.79      0.78      0.78       360
           3       0.84      0.90      0.87       360
           4       0.84      0.73      0.78       360
           5       0.79      0.78      0.78       360

    accuracy                           0.82      2160
   macro avg       0.82      0.83      0.82      2160
weighted avg       0.82      0.82      0.82      2160



model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.151177507061566)}",0.825,0.8240249078492402,0.824763752562418,0.8250000000000001,40


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-emotion-face_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 125.0,
 'days': 0,
 'hours': 0,
 'minutes': 2,
 'seconds': 5.0,
 'text': '0 hari 0 jam 2 menit 5.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 89.0,
 'days': 0,
 'hours': 0,
 'minutes': 1,
 'seconds': 29.0,
 'text': '0 hari 0 jam 1 menit 29.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.151177507061566)}",0.8495,0.8345,0.8304,0.842,0.8345,0.8382,0.8371,0.8379,0.8382,0.4136
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.28942661247167517)}",0.8484,0.8339,0.831,0.8409,0.8362,0.8381,0.8371,0.838,0.8381,0.5152
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.04124626382901348)}",0.8495,0.8339,0.8328,0.8368,0.8356,0.8377,0.8366,0.8375,0.8377,0.4466
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.07896522868499734)}",0.8495,0.8322,0.8333,0.8385,0.8345,0.8376,0.8365,0.8373,0.8376,0.4618
...,...,...,...,...,...,...,...,...,...,...,...
37,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.011253355826007646)}",0.8455,0.8328,0.8258,0.8328,0.8374,0.8348,0.8338,0.835,0.8348,0.4374
38,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.001603718743751331)}",0.8449,0.8345,0.8247,0.8322,0.838,0.8348,0.8338,0.835,0.8348,0.4157
39,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115)}",0.8455,0.8339,0.8258,0.8322,0.8368,0.8348,0.8338,0.835,0.8348,0.441
40,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0008376776400682924)}",0.8449,0.8345,0.8247,0.8316,0.8374,0.8346,0.8335,0.8348,0.8346,0.4459
